# 01 — NR1I2 Atlas Exploration

Inspect the fetched atlas: cell type composition, NR1I2 detection rates, and expression distributions.

**Input**: `data/raw/nr1i2_atlas.h5ad`  
**Output**: summary statistics and QC plots

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import seaborn as sns

from pxr_uncoupling.config import DATA_RAW, NR1I2_SYMBOL, COLOR_CREAM, COLOR_ACCENT

## Load atlas

In [ ]:
adata = ad.read_h5ad(DATA_RAW / "nr1i2_atlas.h5ad")
print(f"{adata.n_obs:,} cells  ×  {adata.n_vars} genes")
print(f"Genes: {adata.var_names.tolist()}")
print(f"\nCell types ({adata.obs['cell_type'].nunique()}):")
display(adata.obs["cell_type"].value_counts().to_frame("n_cells"))

## Cell type composition

In [ ]:
counts = adata.obs["cell_type"].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor(COLOR_CREAM)
ax.set_facecolor(COLOR_CREAM)
counts.plot.barh(ax=ax, color=COLOR_ACCENT, edgecolor="#d0c4b0")
ax.set_xlabel("Number of cells")
ax.set_title("Cell type composition of NR1I2 atlas")
plt.tight_layout()
plt.show()

## NR1I2 detection rate per cell type

Detection = fraction of cells with raw count > 0.

In [ ]:
X = adata.X
if sp.issparse(X):
    X = X.toarray()

nr_idx = list(adata.var_names).index(NR1I2_SYMBOL)
nr_expr = X[:, nr_idx]

detection = (
    pd.DataFrame({"cell_type": adata.obs["cell_type"].values, "nr1i2": nr_expr})
    .groupby("cell_type")["nr1i2"]
    .agg(detection=lambda x: (x > 0).mean(), mean_log1p=lambda x: np.log1p(x).mean())
    .sort_values("detection", ascending=False)
)

display(detection.style.format({"detection": "{:.1%}", "mean_log1p": "{:.3f}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor(COLOR_CREAM)

for ax in axes:
    ax.set_facecolor(COLOR_CREAM)

detection["detection"].plot.barh(ax=axes[0], color=COLOR_ACCENT)
axes[0].set_xlabel("Detection rate (fraction cells > 0)")
axes[0].set_title("NR1I2 detection rate")
axes[0].axvline(0.05, ls="--", color="#999", lw=1, label="5% threshold")
axes[0].legend(fontsize=8)

detection["mean_log1p"].plot.barh(ax=axes[1], color="#8a9a7b")
axes[1].set_xlabel("Mean log1p expression")
axes[1].set_title("NR1I2 mean expression")

plt.tight_layout()
plt.show()

## NR1I2 expression distribution — hepatocyte vs. immune

Hepatocytes show bimodal expression (off/on); immune cells are essentially negative.

In [ ]:
focus_types = ["hepatocyte", "macrophage", "monocyte",
               "CD4-positive, alpha-beta T cell",
               "enterocyte of epithelium of small intestine"]
focus_types = [ct for ct in focus_types if ct in adata.obs["cell_type"].values]

df = pd.DataFrame({
    "cell_type": adata.obs["cell_type"].values,
    "log1p_NR1I2": np.log1p(nr_expr),
})
df = df[df["cell_type"].isin(focus_types)]

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor(COLOR_CREAM)
ax.set_facecolor(COLOR_CREAM)

palette = sns.color_palette("tab10", n_colors=len(focus_types))
for i, ct in enumerate(focus_types):
    vals = df.loc[df["cell_type"] == ct, "log1p_NR1I2"]
    ax.hist(vals, bins=40, alpha=0.55, color=palette[i], label=ct, density=True)

ax.set_xlabel("log1p(NR1I2 counts)")
ax.set_ylabel("Density")
ax.set_title("NR1I2 expression distribution per cell type")
ax.legend(fontsize=7, loc="upper right")
plt.tight_layout()
plt.show()

## Tissue breakdown

In [ ]:
from pxr_uncoupling.config import CELL_TYPE_TISSUE_MAP

adata.obs["tissue"] = adata.obs["cell_type"].map(CELL_TYPE_TISSUE_MAP).fillna("other")
tissue_counts = adata.obs.groupby(["tissue", "cell_type"]).size().reset_index(name="n_cells")
display(tissue_counts.sort_values(["tissue", "n_cells"], ascending=[True, False]))